# Power Map Activation Function Visualization

Visualizing the smooth activation function and the battery power map:

$$\sigma_k(x) = \frac{1}{2}\left(1 + \tanh(kx)\right)$$

$$P_t = P_{\max}\left[\sigma_k(u_t - \epsilon)\sigma_k(0.995 - SOC_t)u_t + \sigma_k(-u_t - \epsilon)\sigma_k(SOC_t - 0.005)u_t\right]$$

In [10]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

k = 500
epsilon = 0.0025
soc_min = 0.005
soc_max = 0.995


def sigma_k(x, k=k):
    return 0.5 * (1 + np.tanh(k * x))


def normalized_power_map(u, soc, k=k, epsilon=epsilon):
    charging_branch = sigma_k(u - epsilon, k) * sigma_k(soc_max - soc, k) * u
    discharging_branch = sigma_k(-u - epsilon, k) * sigma_k(soc - soc_min, k) * u
    return charging_branch + discharging_branch


u = np.linspace(-1, 1, 500)
low_soc = np.linspace(0, 0.02, 350)
high_soc = np.linspace(0.98, 1.0, 350)
U_low, SOC_low = np.meshgrid(u, low_soc)
U_high, SOC_high = np.meshgrid(u, high_soc)
P_low = normalized_power_map(U_low, SOC_low)
P_high = normalized_power_map(U_high, SOC_high)

fig, axes = plt.subplots(
    4,
    1,
    figsize=(8, 15.0),
    gridspec_kw={"height_ratios": [1.0, 1.05, 1.05, 1.05]},
    constrained_layout=True,
)

# Activation gate around zero.
x = np.linspace(-0.02, 0.02, 600)
axes[0].plot(
    x,
    sigma_k(-x - epsilon),
    color="tab:blue",
    linewidth=2.5,
    label=rf"$\sigma_k(-x - \epsilon)$, $\epsilon={epsilon}$",
)
axes[0].plot(
    x,
    sigma_k(x - epsilon),
    color="tab:orange",
    linewidth=2.5,
    label=rf"$\sigma_k(x - \epsilon)$, $\epsilon={epsilon}$",
)
axes[0].axvline(0, color="0.35", linestyle="--", linewidth=1)
axes[0].axvline(epsilon, color="black", linestyle=":", linewidth=1.2)
axes[0].set_title(r"Charging and discharging activation gates")
axes[0].set_xlabel("x")
axes[0].set_ylabel(r"$\sigma_k(\cdot)$")
axes[0].set_ylim(-0.05, 1.05)
axes[0].grid(True, alpha=0.28)
axes[0].legend(loc="lower right", fontsize=8, frameon=False)

# Zoomed power map near the lower SOC boundary.
heatmap = axes[1].pcolormesh(u, low_soc, P_low, shading="auto", cmap="coolwarm", vmin=-1, vmax=1)
axes[1].axvline(-epsilon, color="black", linestyle=":", linewidth=1.2)
axes[1].axvline(epsilon, color="black", linestyle=":", linewidth=1.2)
axes[1].axhline(soc_min, color="black", linestyle=":", linewidth=1.2)
axes[1].set_title(r"Near-empty boundary: discharging is suppressed")
axes[1].set_xlabel(r"control $u_t$")
axes[1].set_ylabel(r"$SOC_t$")

# Zoomed power map near the upper SOC boundary.
axes[2].pcolormesh(u, high_soc, P_high, shading="auto", cmap="coolwarm", vmin=-1, vmax=1)
axes[2].axvline(-epsilon, color="black", linestyle=":", linewidth=1.2)
axes[2].axvline(epsilon, color="black", linestyle=":", linewidth=1.2)
axes[2].axhline(soc_max, color="black", linestyle=":", linewidth=1.2)
axes[2].set_title(r"Near-full boundary: charging is suppressed")
axes[2].set_xlabel(r"control $u_t$")
axes[2].set_ylabel(r"$SOC_t$")
fig.colorbar(heatmap, ax=axes[1:3], label=r"$P_t / P_{\max}$")

# Effect of k and epsilon on the shifted activation gate.
x_zoom = np.linspace(-0.02, 0.02, 800)
for k_value in [50, 150, 500, 1500]:
    axes[3].plot(
        x_zoom,
        sigma_k(x_zoom - epsilon, k=k_value),
        linewidth=2,
        label=rf"$k={k_value}$, $\epsilon={epsilon}$",
    )
for epsilon_value in [0.0, 0.005, 0.01]:
    axes[3].plot(
        x_zoom,
        sigma_k(x_zoom - epsilon_value, k=k),
        linestyle="--",
        linewidth=2,
        label=rf"$k={k}$, $\epsilon={epsilon_value}$",
    )
axes[3].axvline(0, color="0.35", linestyle="--", linewidth=1)
axes[3].axvline(epsilon, color="black", linestyle=":", linewidth=1)
axes[3].set_title(r"Effect of $k$ and $\epsilon$ on $\sigma_k(x - \epsilon)$")
axes[3].set_xlabel("x")
axes[3].set_ylabel(r"$\sigma_k(x - \epsilon)$")
axes[3].set_ylim(-0.05, 1.05)
axes[3].grid(True, alpha=0.28)
axes[3].legend(loc="upper center", bbox_to_anchor=(0.5, -0.22), fontsize=8, frameon=False, ncol=2)

fig.suptitle(f"Smooth charge/discharge gating (k={k})", fontsize=14)

output_path = Path.cwd() / "power_map_visualization.png"
if Path.cwd().name != "notebooks" and (Path.cwd() / "notebooks").exists():
    output_path = Path.cwd() / "notebooks" / "power_map_visualization.png"
fig.savefig(output_path, dpi=200, bbox_inches="tight")
plt.show()

output_path.resolve()


WindowsPath('C:/Users/thap_as/Documents/THESIS/notebooks/power_map_visualization.png')